In [ ]:
# Install LangGraph — the core library we'll use to build graph-based AI workflows.
# The -q flag suppresses verbose installation output.
!pip install -q langgraph

In [ ]:
import operator  # Used for the 'add' reducer — it tells LangGraph to append log lists instead of overwriting

from IPython.display import Image          # Displays images (graph diagrams) inline in Jupyter
from langchain_core.runnables import Runnable  # Base type for compiled LangGraph graphs
from langgraph.graph import END, START, StateGraph  # Core LangGraph building blocks:
                                                    # START/END are special sentinel nodes,
                                                    # StateGraph is the graph builder class
from pathlib import Path                   # Cross-platform file path handling
from typing import Annotated, TypedDict    # TypedDict: defines the shape of the state dict
                                           # Annotated: attaches metadata (like reducers) to fields


# Helper function: renders the compiled graph as a PNG image and shows it inline in Jupyter.
# 'runnable' is the compiled LangGraph graph object.
# 'output_png' is the file path where the image will be saved temporarily.
def display_graph(runnable: Runnable, output_png: Path):
    with output_png.open(mode="wb") as file:
        # draw_mermaid_png() converts the graph structure into a visual Mermaid diagram
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

In [ ]:
# The 'State' is the shared data object that travels through every node in the graph.
# TypedDict defines it as a plain dictionary with specific typed fields —
# no magic, just a schema that tells LangGraph (and your IDE) what data to expect.
class TextProcessingState(TypedDict):
    raw_text: str          # The original unprocessed input text
    clean_text: str        # Lowercased and trimmed version produced by the 'normalize' node
    word_count: int        # Total number of words, counted by the 'count' node
    tokens: list[str]      # List of individual alphabetic words from the 'tokenize' node

    # Annotated with 'operator.add' means when multiple nodes return a new 'log' list,
    # LangGraph will automatically CONCATENATE them instead of overwriting.
    # This lets every node append its own log entry without clobbering previous ones.
    log: Annotated[list[str], operator.add]

In [ ]:
# Each function below is a graph NODE.
# A node receives the FULL current state and returns a PARTIAL dict
# (only the fields it wants to update). LangGraph merges the partial dict into the state.

def normalize(state: TextProcessingState):
    # Step 1: Clean the raw text — lowercase everything and strip whitespace
    cleaned = state["raw_text"].lower().strip()
    # Return only updated fields; other state fields are left unchanged
    return { "clean_text": cleaned, "log": [f"normalized ({len(cleaned)} chars)"] }

def tokenize(state: TextProcessingState):
    # Step 2: Split the cleaned text into words (tokens)
    # .isalpha() filters out punctuation, numbers, and symbols — keeps only pure words
    tokens = [t for t in state["clean_text"].split() if t.isalpha()]
    return { "tokens": tokens, "log": [f"tokenized ({len(tokens)} tokens)"] }

def count_words(state: TextProcessingState) -> dict:
    # Step 3: Count how many valid tokens (words) we have
    return { "word_count": len(state["tokens"]), "log": [f"counted: {len(state['tokens'])} words"] }

def report(state: TextProcessingState) -> dict:
    # Step 4 (final node): Print a human-readable summary of the full pipeline results
    print("===== REPORT =====")
    print(f"Original     : {state['raw_text']!r}")
    print(f"Normalized   : {state['clean_text']!r}")
    print(f"Word count   : {state['word_count']}")

    print("Log:")
    for line in state["log"]:
        print("- ", line)

    # Append a final entry to confirm this node ran
    return { "log": ["reported"] }

In [ ]:
# Build the graph by registering nodes and wiring them together with edges.
# StateGraph takes the state schema so it knows what data shape flows through the graph.
graph_builder = StateGraph(TextProcessingState)

# Register each node — the first argument is the node's name (used in edge definitions)
graph_builder.add_node("normalize", normalize)
graph_builder.add_node("tokenize", tokenize)
graph_builder.add_node("count", count_words)
graph_builder.add_node("report", report)

# Define the execution order with edges.
# This is a simple LINEAR (sequential) pipeline — each node runs one after the other.
# START and END are special sentinel nodes that mark the entry and exit points.
graph_builder.add_edge(START, "normalize")
graph_builder.add_edge("normalize", "tokenize")
graph_builder.add_edge("tokenize", "count")
graph_builder.add_edge("count", "report")
graph_builder.add_edge("report", END)

# Compile the graph — validates the structure and returns a runnable object
graph = graph_builder.compile()

In [ ]:
# Render and display the graph as a visual diagram.
# This is just for visualization — it doesn't run the graph.
# You'll see boxes for each node connected by arrows showing the execution order.
display_graph(graph, Path("/content/graph.png"))

In [ ]:
# Run the graph by providing an initial state with just the 'raw_text' field.
# LangGraph will pass this state through each node in order,
# and each node will fill in the remaining fields (clean_text, tokens, word_count, log).
final_state = graph.invoke(
    input={
        "raw_text": "Hello, world! Today we are talking about LangGraph and how to build multi-agent systems."
    }
)

In [ ]:
# Inspect the final state after the graph has finished running.
# You should see all fields populated: clean_text, tokens, word_count, and the full log list.
final_state